# SWE-Gym 32B verifier on SWE-bench Lite runs (TraceTrust, E2)

Scores the runs in `verifier_prompts_lite.jsonl` with `SWE-Gym/OpenHands-32B-Verifier` (4-bit GGUF,
`mradermacher/OpenHands-32B-Verifier-GGUF`, Q4_K_M) and writes `p(YES)` per run.

The prompts were rendered locally by `python -m experiments.verifier_prompts`, in exactly the verifier's
training format (checked 50/50 against its training data). These runs come from repositories the verifier
never trained on, so this is a fair comparison.

**Setup**
1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add Data: upload `data/verifier_prompts_lite.jsonl` as a Kaggle dataset and attach it.
3. Run all. Results are appended to `/kaggle/working/verifier_scores_lite.csv` after every run.
4. If the session ends early: save the version, then start a new session with that version's output
   attached as an input. Finished runs are skipped.
5. Download `verifier_scores_lite.csv` into the repo's `data/` folder.

In [ ]:
# Build llama.cpp with CUDA for the T4 (compute capability 7.5); about 15-30 min. No prebuilt GPU wheel
# matches Kaggle's Python, and a plain pip install silently builds a CPU-only version. On failure the
# end of the build log is printed and the full log is saved to /kaggle/working/llama_build.log.
# GGML_CUDA_NO_VMM: Kaggle's image has no libcuda to link against at build time.
import glob, os, shutil, subprocess, sys
print(sys.version)
nvcc = shutil.which("nvcc") or next(iter(sorted(glob.glob("/usr/local/cuda*/bin/nvcc"))), None)
print("nvcc:", nvcc)
if nvcc:
    print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1])
env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_CUDA_NO_VMM=ON", FORCE_CMAKE="1")
if nvcc:
    env["CUDACXX"] = nvcc
    env["PATH"] = os.path.dirname(nvcc) + ":" + env["PATH"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--verbose", "llama-cpp-python"],
                   env=env, capture_output=True, text=True)
log = r.stdout + r.stderr
open("/kaggle/working/llama_build.log", "w").write(log)
if r.returncode:
    print(log[-8000:])
    raise RuntimeError("llama-cpp-python build failed; the end of the log is printed above")
import llama_cpp
assert llama_cpp.llama_supports_gpu_offload(), "llama.cpp was built without GPU support"
print("llama.cpp", llama_cpp.__version__, "with GPU offload")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, json, math, os, time
import numpy as np
import pandas as pd

MODEL_REPO = "mradermacher/OpenHands-32B-Verifier-GGUF"
MODEL_FILE = "OpenHands-32B-Verifier.Q4_K_M.gguf"   # 19.9 GB
N_CTX = 32768          # SWE-Gym trained with 32k-token inputs; 24576 or 16384 if memory/time is short
KV_Q8 = True           # 8-bit KV cache so 32k fits on 2 x T4; needs FLASH_ATTN
FLASH_ATTN = True      # if loading fails on T4, set both to False and N_CTX = 24576
MAX_HOURS = 11.0       # stop cleanly before Kaggle's 12 h limit
OUT = "/kaggle/working/verifier_scores_lite.csv"

PROMPTS = glob.glob("/kaggle/input/**/verifier_prompts_lite.jsonl", recursive=True)
assert PROMPTS, "attach the dataset holding verifier_prompts_lite.jsonl"
recs = [json.loads(l) for l in open(PROMPTS[0], encoding="utf-8")]
print(len(recs), "runs to score")

In [ ]:
from huggingface_hub import hf_hub_download
t = time.time()
model_path = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir="/tmp/verifier")
print(model_path, f"{os.path.getsize(model_path) / 1e9:.1f} GB in {time.time() - t:.0f}s")

In [ ]:
from llama_cpp import Llama
GGML_TYPE_F16, GGML_TYPE_Q8_0 = 1, 8
llm = Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=N_CTX, n_batch=512, tensor_split=[0.5, 0.5],
            flash_attn=FLASH_ATTN, type_k=GGML_TYPE_Q8_0 if KV_Q8 else GGML_TYPE_F16,
            type_v=GGML_TYPE_Q8_0 if KV_Q8 else GGML_TYPE_F16, logits_all=False, verbose=False)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
YES = llm.tokenize(b"YES", add_bos=False, special=False)
NO = llm.tokenize(b"NO", add_bos=False, special=False)
assert len(YES) == 1 and len(NO) == 1, (YES, NO)
YES, NO = YES[0], NO[0]

def chat(judge_system: str, user: str) -> str:
    # Qwen2.5 chat template; the answer is prefilled up to "<judgement>" so the next token is YES or NO
    return (f"<|im_start|>system\n{judge_system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n<judgement>")

def ntok(s: str) -> int:
    return len(llm.tokenize(s.encode("utf-8"), add_bos=False, special=True))

def fit(rec: dict) -> tuple[str, int]:
    """Prompt within N_CTX: keep the first turn (the issue) and as many final turns as fit.
    -> (prompt, number of middle turns dropped)"""
    turns = rec["turns"]
    budget = N_CTX - 64 - ntok(chat(rec["judge_system"], rec["head"] + rec["tail"]))
    sizes = [ntok(t) for t in turns]
    if sum(sizes) <= budget:
        keep = turns
    else:
        budget -= sizes[0]
        tail, i = [], len(turns) - 1
        while i > 0 and sizes[i] <= budget:
            budget -= sizes[i]
            tail.append(turns[i])
            i -= 1
        keep = [turns[0]] + tail[::-1]
    return chat(rec["judge_system"], rec["head"] + "".join(keep) + rec["tail"]), len(turns) - len(keep)

def last_logits(prompt: str) -> tuple[np.ndarray, int]:
    toks = llm.tokenize(prompt.encode("utf-8"), add_bos=False, special=True)
    llm.reset()
    llm.eval(toks)
    return np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).copy(), len(toks)

def score(rec: dict) -> dict:
    prompt, dropped = fit(rec)
    logits, n = last_logits(prompt)
    lse = np.logaddexp.reduce(logits.astype(np.float64))
    lp_yes, lp_no = logits[YES] - lse, logits[NO] - lse
    return {"uid": rec["uid"], "run": rec["run"], "version": rec["version"],
            "p_yes": float(1 / (1 + math.exp(lp_no - lp_yes))),       # renormalised over {YES, NO}
            "lp_yes": float(lp_yes), "lp_no": float(lp_no), "mass_yes_no": float(math.exp(lp_yes) + math.exp(lp_no)),
            "n_tokens": n, "turns_dropped": dropped}

In [ ]:
# Self-test: the logits read back must agree with llama.cpp's own greedy next token.
p = chat("Answer with one word.", "Is the sky blue on a clear day? Respond with YES or NO.")
logits, _ = last_logits(p)
greedy = llm.create_completion(p, max_tokens=1, temperature=0.0)["choices"][0]["text"]
print("argmax:", repr(llm.detokenize([int(np.argmax(logits))]).decode()), "| greedy:", repr(greedy))
assert llm.detokenize([int(np.argmax(logits))]).decode().strip() == greedy.strip()
r = score(recs[0]); print(r)
assert r["mass_yes_no"] > 0.5, "YES/NO should carry most of the probability after '<judgement>'"

In [ ]:
done = set()
for f in glob.glob("/kaggle/input/**/verifier_scores_lite.csv", recursive=True) + ([OUT] if os.path.exists(OUT) else []):
    prev = pd.read_csv(f)
    done |= set(prev["uid"])
    if f != OUT:
        prev.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
print(len(done), "already scored")

t0 = time.time()
todo = [r for r in recs if r["uid"] not in done]
for i, rec in enumerate(todo, 1):
    if (time.time() - t0) / 3600 > MAX_HOURS:
        print("time limit reached; save this version and resume in a new session"); break
    t = time.time()
    row = score(rec) | {"seconds": round(time.time() - t, 1)}
    pd.DataFrame([row]).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    eta = (time.time() - t0) / i * (len(todo) - i) / 3600
    print(f"{i}/{len(todo)} {rec['uid']} p_yes={row['p_yes']:.3f} tokens={row['n_tokens']} "
          f"dropped={row['turns_dropped']} {row['seconds']}s  eta {eta:.1f} h")

In [ ]:
s = pd.read_csv(OUT).drop_duplicates("uid")
print(len(s), "scored;", s.groupby("version")["p_yes"].describe().round(3))